In [1]:
import sys
sys.executable

'/home/gmvaz/Installs/miniforge3/envs/gems/bin/python'

In [29]:
# core
import numpy as np
import pandas as pd

# COBRApy
from cobra.core.configuration import Configuration
from cobra.io import load_model

#iMATpy
from imatpy.imat import imat
from imatpy.model_utils import read_model
from imatpy.parse_gpr import gene_to_rxn_weights
from imatpy.model_creation import generate_model, simple_bounds_model, subset_model

In [ ]:
# set solver to glpk, this is the default optimizer installed with cobra so it is the easiest one to set for imat since we it's already installed
Configuration().solver = "glpk"

In [34]:
# load the test ecoli model in cobrapy 
model = load_model("textbook")
model

Name,e_coli_core
Memory address,7c5b37e13b10
Number of metabolites,72
Number of reactions,95
Number of genes,137
Number of groups,0
Objective expression,1.0*Biomass_Ecoli_core - 1.0*Biomass_Ecoli_core_reverse_2cdba
Compartments,"cytosol, extracellular"


In [ ]:
# get gene IDs from our test model 
len(model.genes)
[g.id for g in model.genes][:20]

In [21]:
# sanity check the gene id by seeing which reaction a gene maps
test_gene1 = model.genes.get_by_id("b1241")
test_gene1.reactions

frozenset({<Reaction ACALD at 0x7c5b453988d0>,
           <Reaction ALCD2x at 0x7c5b4539dd90>})

In [22]:
# see the GPR associated with the reactions for that gene 
for r in test_gene1.reactions:
    print(r.id, "GPR:", r.gene_reaction_rule)

ALCD2x GPR: b1478 or b0356 or b1241
ACALD GPR: b0351 or b1241


In [23]:
# A pandas series of gene expression weights
# The gene IDs need to match the actual model.gene IDs
model_weights = pd.Series({
           "b1241": 1,
           "b0351": -1,
           "s0001": -1,
           "b3115": 0,
           "b1849": -1,
           "b2296": 0,
           "b1276": 0,
           "b0118": -1,
           "b0474": 0,
           "b0116": 1,
       })

In [37]:
help(load_model)

Help on function load_model in module cobra.io.web.load:

load_model(model_id: str, repositories: Iterable[cobra.io.web.abstract_model_repository.AbstractModelRepository] = (<cobra.io.web.cobrapy_repository.Cobrapy object at 0x7c5b45281a10>, <cobra.io.web.bigg_models_repository.BiGGModels object at 0x7c5b45281d50>, <cobra.io.web.biomodels_repository.BioModels object at 0x7c5b453a2550>), cache: bool = True) -> 'Model'
    Download an SBML model from a remote repository.
    
    Downloaded SBML documents are by default stored in a cache on disk such that future
    access is much faster. By default, models can be loaded from the following
    repositories:
    
    * BiGG Models
    * BioModels
    
    You can use the ``AbstractModelRepository`` class as a parent to implement your own
    repository accessor which you pass to the ``load_model`` function. In case you
    implement a new interface, please consider submitting a pull request to COBRApy.
    
    Parameters
    ----------
 

In [24]:
# Reindex list of gene IDs, keep the gene weights labeled in the pandas series, and additional added genes will get weight zero
gene_weights_all = model_weights.reindex([g.id for g in model.genes], fill_value=0)

# Use the expanded vector here to get reaction weights 
reaction_weights = gene_to_rxn_weights(model=model, gene_weights=gene_weights_all)

print(reaction_weights.head())

ACALD     1.0
ACALDt   -1.0
ACKr      0.0
ACONTa    0.0
ACONTb    0.0
dtype: float64


In [25]:
# run imat
imat_results = imat(
    model=model,
    rxn_weights=reaction_weights,
    epsilon=1,
    threshold=0.01
)

In [26]:
# Print the imat objective
print(f"iMAT Objective: {imat_results.objective_value}")
print("Number of active reactions:",
      (imat_results.fluxes.abs() > 1e-6).sum())

iMAT Objective: 5.0
Number of active reactions: 56


In [15]:
# Print the imat flux distribution
print(f"iMAT Flux Distribution: \n{imat_results.fluxes}")

iMAT Flux Distribution: 
ACALD    -8.488023
ACALDt    0.000000
ACKr     -8.284425
ACONTa    0.247695
ACONTb    0.247695
            ...   
TALA     -0.037756
THD2      3.618581
TKT1     -0.037756
TKT2     -0.113943
TPI       9.790075
Name: fluxes, Length: 95, dtype: float64


In [18]:
# Generate a new model based on the iMAT results
#??
new_model = generate_model(model=model, rxn_weights=reaction_weights, method="fva")

In [31]:
set_bounds_model = simple_bounds_model(model=model, rxn_weights=reaction_weights, epsilon=1, threshold=0.01)

In [32]:
rmv_inactive_rxns_model = subset_model(model=model, rxn_weights=reaction_weights, epsilon=1, threshold=0.01)